# AgriAgent: Gemma 4 Native Function Calling Demo

This notebook demonstrates how **Gemma 4**'s native function calling capabilities are utilized in the **AgriAgent** system to integrate IoT soil sensors, localized weather forecasting, and market crop rates. This serves as the **Clonable Notebook** submission option for the TFUG Prayagraj Hackathon.

### System Architecture
1. **User Query**: Farmer asks a question in plain natural language (e.g., "Water my wheat field if moisture is low").
2. **Gemma 4 Thought**: Model analyzes the prompt and decides to fetch sensor telemetry and weather data.
3. **Tool Dispatching**: Model issues a structured tool call payload.
4. **API Execution**: The local system parses the tool call, fetches sensor outputs, and sends the raw telemetry back to the model.
5. **Final Output**: Gemma 4 synthesizes the data and generates a clear, actionable agricultural response.

## 1. Defining Agricultural Tool APIs
Below are the mock APIs simulating IoT farm sensors, irrigation actuators, and market prices.

In [ ]:
import json
from datetime import datetime

# Mock database representing three fields in Prayagraj
FIELDS_DB = {
    "field-a": {
        "name": "North Field",
        "crop": "Rice",
        "moisture": 72,
        "ph": 6.2,
        "npk": {"N": 45, "P": 30, "K": 55}
    },
    "field-b": {
        "name": "West Terrace",
        "crop": "Wheat",
        "moisture": 34, # Low moisture -> Needs water
        "ph": 6.8,
        "npk": {"N": 20, "P": 15, "K": 40}
    }
}

def get_soil_sensors(field_id: str) -> str:
    """Fetches real-time moisture %, temperature, and NPK levels for a field."""
    field_id = field_id.lower().strip()
    if field_id in FIELDS_DB:
        data = FIELDS_DB[field_id]
        return json.dumps({
            "status": "success",
            "field_id": field_id,
            "data": data,
            "timestamp": datetime.now().isoformat()
        })
    return json.dumps({"status": "error", "message": f"Field {field_id} not found."})

def trigger_irrigation(field_id: str, duration_minutes: int) -> str:
    """Triggers the field smart irrigation sprinkler system."""
    field_id = field_id.lower().strip()
    if field_id in FIELDS_DB:
        prev_moisture = FIELDS_DB[field_id]["moisture"]
        new_moisture = min(100, prev_moisture + (duration_minutes * 2))
        FIELDS_DB[field_id]["moisture"] = new_moisture
        return json.dumps({
            "status": "success",
            "message": f"Irrigation valve active for {duration_minutes} mins. Moisture increased from {prev_moisture}% to {new_moisture}%."
        })
    return json.dumps({"status": "error", "message": f"Field {field_id} not found."})

def get_weather_forecast(field_id: str) -> str:
    """Fetches local weather forecasts and severe rain warnings."""
    # Mock forecast for the area
    return json.dumps({
        "location": "Prayagraj West",
        "precip_probability": 10, # low probability
        "condition": "Dry & Sunny",
        "alerts": []
    })

## 2. System Prompts & Gemma 4 Tool Declarations
We declare the tools list as a structured JSON object so Gemma 4 can construct valid calls.

In [ ]:
SYSTEM_PROMPT = """
You are AgriAgent, an autonomous agricultural AI assistant powered by Gemma 4.
You help local farmers manage crop yields, moisture, and fertilizers.

You have access to the following tools. To call them, you must respond with a thought process and then output a markdown code block of type `json` containing the tool call.

Tools Available:
1. get_soil_sensors(field_id: string)
2. trigger_irrigation(field_id: string, duration_minutes: int)
3. get_weather_forecast(field_id: string)

Example Output format for a tool call:
Thinking: The user wants to check sensor telemetry. I will query the sensors for field-b first.
```json
{
    "tool_calls": [
        {
            "name": "get_soil_sensors",
            "arguments": {"field_id": "field-b"}
        }
    ]
}
```
"""

## 3. Simulating Gemma 4 Function Calling Execution Loop
Here we define the driver function that executes the model's call, invokes the python API, and prints out the progress.

In [ ]:
def simulate_gemma4_call(user_query: str):
    print(f"[USER QUERY]: {user_query}\n")
    
    # --- Gemma Step 1: Receives Query, Decides Tool Call ---
    print("--- STEP 1: Gemma 4 Thought & JSON Generation ---")
    # Simulating model generating response
    if "water" in user_query or "irrigate" in user_query or "moisture" in user_query:
        thought = "The farmer is asking to water or check moisture on Field B. I need to fetch the soil sensor details to inspect current moisture levels."
        tool_call = {
            "tool_calls": [
                {
                    "name": "get_soil_sensors",
                    "arguments": {"field_id": "field-b"}
                }
            ]
        }
    else:
        print("No tool call required. Direct chat.")
        return
        
    print(f"Thinking: {thought}")
    print(f"Generated Tool Call JSON:\n{json.dumps(tool_call, indent=4)}\n")
    
    # --- System Step 2: Executes API Call ---
    print("--- STEP 2: Local System Tool Execution ---")
    call_info = tool_call["tool_calls"][0]
    fn_name = call_info["name"]
    fn_args = call_info["arguments"]
    
    if fn_name == "get_soil_sensors":
        api_response = get_soil_sensors(fn_args["field_id"])
    else:
        api_response = "{}"
        
    print(f"API \`{fn_name}\` Returned: {api_response}\n")
    
    # --- Gemma Step 3: Second Model Turn (Irrigation Decision) ---
    print("--- STEP 3: Gemma 4 Evaluates Tool Output & Decides Actions ---")
    sensor_data = json.loads(api_response)["data"]
    moisture = sensor_data["moisture"]
    
    print(f"Thinking: Moisture level is {moisture}%. This is low (optimal is > 50%). I will check the weather forecast to make sure rain is not coming.")
    weather_response = get_weather_forecast("field-b")
    weather_data = json.loads(weather_response)
    print(f"API \`get_weather_forecast\` Returned: {weather_response}")
    
    print(f"Thinking: Rain probability is only {weather_data['precip_probability']}%. I should proceed with open irrigation valve.")
    irr_call = {
        "tool_calls": [
            {
                "name": "trigger_irrigation",
                "arguments": {"field_id": "field-b", "duration_minutes": 20}
            }
        ]
    }
    print(f"Generated Secondary Tool Call JSON:\n{json.dumps(irr_call, indent=4)}\n")
    
    # --- System Step 4: Executes Irrigation ---
    print("--- STEP 4: Local System Sprinkler Activation ---")
    irr_args = irr_call["tool_calls"][0]["arguments"]
    irr_response = trigger_irrigation(irr_args["field_id"], irr_args["duration_minutes"])
    print(f"API \`trigger_irrigation\` Returned: {irr_response}\n")
    
    # --- Gemma Step 5: Final Synthesis ---
    print("--- STEP 5: Gemma 4 Final Synthesis Response ---")
    final_answer = (
        f"I checked the sensors for West Terrace (Field B). The soil moisture was low ({moisture}%). "
        f"Since the weather forecast shows a dry day ahead, I have successfully opened the irrigation sprinkler valve for 20 minutes. "
        f"The soil moisture level has been successfully restored to {FIELDS_DB['field-b']['moisture']}%."
    )
    print(f"[AGRIAGENT FINAL RECOMMENDATION]:\n{final_answer}")

## 4. Run Simulation
Let's execute the complete pipeline simulation for a farmer query.

In [ ]:
simulate_gemma4_call("Check moisture in Field B and water if needed")